In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN
import re
from sklearn.metrics import silhouette_score
from sklearn.model_selection import ParameterGrid

from bs4 import BeautifulSoup
import re

from spacy.lang.en.stop_words import STOP_WORDS
import en_core_web_sm

In [2]:
import os 
print(os.getcwd())
print(os.listdir())

c:\Users\LN\Documents\Jedha\Projets\NORTH_FACE\Helene
['.gitignore', '01-North_Face_ecommerce.ipynb', '02-North-Face_DBSCAN.ipynb', '03-North-Face_wordcloud.ipynb', '04-North-Face_Truncate_SVD.ipynb', 'data', 'data_cleaning.py', 'dbscan_cleaned.ipynb', 'groups_of_products.ipynb', 'iframe_figures', 'preprocessing.ipynb']


In [3]:
df = pd.read_csv(r'C:\Users\LN\Documents\Jedha\Projets\NORTH_FACE\Helene\data\raw.csv')

In [4]:
df.head(1)

,id,description
0,1,Active classic boxers - There's a reason why o...


In [5]:
# sample = data.sample(n=5, random_state=42)
sample = df.copy()

In [6]:
pd.set_option('display.max_colwidth', None)
display(sample)

,id,description
0,1,"Active classic boxers - There's a reason why our boxers are a cult favorite - they keep their cool, especially in sticky situations. The quick-drying, lightweight underwear takes up minimal space in a travel pack. An exposed, brushed waistband offers next-to-skin softness, five-panel construction with a traditional boxer back for a classic fit, and a functional fly. Made of 3.7-oz 100% recycled polyester with moisture-wicking performance. Inseam (size M) is 4 1/2"". Recyclable through the Common Threads Recycling Program.<br><br><b>Details:</b><ul> <li>""Silky Capilene 1 fabric is ultralight, breathable and quick-to-dry""</li> <li>""Exposed, brushed elastic waistband for comfort""</li> <li>5-panel construction with traditional boxer back</li> <li>""Inseam (size M) is 4 1/2""""""</li></ul><br><br><b>Fabric: </b>3.7-oz 100% all-recycled polyester with Gladiodor natural odor control for the garment. Recyclable through the Common Threads Recycling Program<br><br><b>Weight: </b>99 g (3.5 oz)<br><br>Made in Mexico."
1,2,"Active sport boxer briefs - Skinning up Glory requires enough movement without your boxers deciding to poach their own route. The form-fitting Active Sport Boxer Briefs are made from breathable 93% polyester (71% recycled) fabric that's fast-wicking, dries quickly and has 7% spandex for stretch; the seamless waistband and soft leg edges won't roll or bind. The gusseted, flat-sewn 6"" inseam (size M) is offset to prevent inner-thigh chafe. Fly-free with a smooth front panel. Recyclable through the Common Threads Recycling Program.<br><br><b>Details:</b><ul> <li>""Stretch mesh provides support, open-weave mesh for airflow, wicks efficiently, dries fast""</li> <li>Seamless construction</li> <li>""Flat-sewn, gusseted inseam is set forward to prevent inner-thigh chafe""</li> <li>Fly-free support</li> <li>""Inseam (size M) is 6""""""</li></ul><br><br><b>Fabric: </b>""4.6-oz 93% polyester (71% recycled)/7% spandex, with moisture-wicking performance. Recyclable through the Common Threads Recycling Program""<br><br><b>Weight: </b>(60 g 2.1 oz)<br><br>Made in Israel."
2,3,"Active sport briefs - These superbreathable no-fly briefs are the minimalist's choice for high-octane endeavors. Made from a blend of fast-wicking, quick-drying 93% polyester (71% recycled) and 7% spandex that has both stretch-mesh (for support) and open mesh (for cooling airflow). Soft edging at the leg openings and a seamless waist won't roll or create friction against layers. With a smooth front panel for opacity. Recyclable through the Common Threads Recycling Program.<br><br><b>Details:</b><ul> <li>""Stretch mesh provides support, open-weave mesh for airflow, wicks efficiently and dries fast""</li> <li>Seamless construction</li> <li>Leg openings have half-inch self-binding for comfort</li> <li>Fly-free support</li></ul><br><br><b>Fabric: </b>""4.6-oz 93% polyester (71% recycled)/7% spandex, with moisture-wicking performance. Recyclable through the Common Threads Recycling Program""<br><br><b>Weight: </b>(49 g 1.7 oz)<br><br>Made in Israel."
3,4,"Alpine guide pants - Skin in, climb ice, switch to rock, traverse a knife-edge ridge and boogie back down - these durable, weather-resistant and breathable soft-shell pants keep stride on every mountain endeavor. The midweight stretch-woven polyester won't restrict your moves, and the brushed interior maintains next-to-skin comfort. A 2-way zippered fly keeps things easy with a harness on, and the gusseted, zippered cuffs have 2-position snaps and tie-down loops. With water-resistant zippered pockets: two front, two thigh, one back hip. Recyclable through the Common Threads Recycling Program.<br><br><b>Details:</b><ul> <li>Durable stretch-woven polyester (47% recycled) with DWR (durable water repellent) finish is water- and wind-resistant and highly breathable</li> <li>Brushed interior for next-to-skin comfort</li> <li>Waistband has belt loops and elastic on back</li> <li>2-way zippered fly

In [7]:
sample_clean = sample.copy()

# Créer un objet BeautifulSoup pour chaque description
html = sample_clean["description"].fillna("").apply(
    lambda text: BeautifulSoup(text, "html.parser")
)

In [8]:
def extract_description(soup):
    """Récupérer le texte avant la première balise <br> ou <b>."""
    chunks = []

    for element in soup.contents:
        if element.name in ["br", "b"]:
            break

        chunks.append(str(element))

    text_html = " ".join(chunks)
    return BeautifulSoup(text_html, "html.parser").get_text(" ", strip=True)


sample_clean["description_text"] = html.apply(extract_description)

In [ ]:
def extract_advantages(soup):
    texte = BeautifulSoup(text_html, "html.parser").get_text(" ", strip=True)

    # begin = texte.index("- they keep")
    # end_pointer = "functional fly."
    # end = texte.index(marqueur_fin, debut) + len(marqueur_fin)

    # description = texte[debut:fin]

In [9]:
def extract_details(soup):
    """Récupérer les éléments de la liste Details."""
    title = soup.find("b", string=re.compile(r"^\s*Details\s*:\s*$", re.I))

    if title is None:
        return []

    list = title.find_next_sibling()

    if list is None or list.name != "ul":
        return []

    details = []

    for element in list.find_all("li"):
        details.append(element.get_text(" ", strip=True))

    return details


sample_clean["details"] = html.apply(extract_details)

In [10]:
def extract_value(soup, nom):
    """Récupérer le texte qui suit un titre en gras."""
    pattern = rf"^\s*{re.escape(nom)}\s*:\s*$"
    title = soup.find("b", string=re.compile(pattern, re.I))

    if title is None:
        return None

    chunks = []

    for element in title.next_siblings:
        if element.name in ["br", "b"]:
            break

        chunks.append(str(element))

    text_html = " ".join(chunks)
    return BeautifulSoup(text_html, "html.parser").get_text(" ", strip=True)

sample_clean["fabric"] = html.apply(lambda soup: extract_value(soup, "Fabric"))
sample_clean["weight"] = html.apply(lambda soup: extract_value(soup, "Weight"))

In [11]:
def extract_country(soup):
    """Récupérer le pays après « Made in »."""
    for text in soup.stripped_strings:
        if text.lower().startswith("made in "):
            return text[8:].strip().rstrip(".")

    return None

sample_clean["country"] = html.apply(extract_country)

In [12]:
display(sample_clean[[
    "description_text",
    "details",
    "fabric",
    "weight",
    "country"
]])

,description_text,details,fabric,weight,country
0,"Active classic boxers - There's a reason why our boxers are a cult favorite - they keep their cool, especially in sticky situations. The quick-drying, lightweight underwear takes up minimal space in a travel pack. An exposed, brushed waistband offers next-to-skin softness, five-panel construction with a traditional boxer back for a classic fit, and a functional fly. Made of 3.7-oz 100% recycled polyester with moisture-wicking performance. Inseam (size M) is 4 1/2"". Recyclable through the Common Threads Recycling Program.","[""Silky Capilene 1 fabric is ultralight, breathable and quick-to-dry"", ""Exposed, brushed elastic waistband for comfort"", 5-panel construction with traditional boxer back, ""Inseam (size M) is 4 1/2""""""]",3.7-oz 100% all-recycled polyester with Gladiodor natural odor control for the garment. Recyclable through the Common Threads Recycling Program,99 g (3.5 oz),Mexico
1,"Active sport boxer briefs - Skinning up Glory requires enough movement without your boxers deciding to poach their own route. The form-fitting Active Sport Boxer Briefs are made from breathable 93% polyester (71% recycled) fabric that's fast-wicking, dries quickly and has 7% spandex for stretch; the seamless waistband and soft leg edges won't roll or bind. The gusseted, flat-sewn 6"" inseam (size M) is offset to prevent inner-thigh chafe. Fly-free with a smooth front panel. Recyclable through the Common Threads Recycling Program.","[""Stretch mesh provides support, open-weave mesh for airflow, wicks efficiently, dries fast"", Seamless construction, ""Flat-sewn, gusseted inseam is set forward to prevent inner-thigh chafe"", Fly-free support, ""Inseam (size M) is 6""""""]","""4.6-oz 93% polyester (71% recycled)/7% spandex, with moisture-wicking performance. Recyclable through the Common Threads Recycling Program""",(60 g 2.1 oz),Israel
2,"Active sport briefs - These superbreathable no-fly briefs are the minimalist's choice for high-octane endeavors. Made from a blend of fast-wicking, quick-drying 93% polyester (71% recycled) and 7% spandex that has both stretch-mesh (for support) and open mesh (for cooling airflow). Soft edging at the leg openings and a seamless waist won't roll or create friction against layers. With a smooth front panel for opacity. Recyclable through the Common Threads Recycling Program.","[""Stretch mesh provides support, open-weave mesh for airflow, wicks efficiently and dries fast"", Seamless construction, Leg openings have half-inch self-binding for comfort, Fly-free support]","""4.6-oz 93% polyester (71% recycled)/7% spandex, with moisture-wicking performance. Recyclable through the Common Threads Recycling Program""",(49 g 1.7 oz),Israel
3,"Alpine guide pants - Skin in, climb ice, switch to rock, traverse a knife-edge ridge and boogie back down - these durable, weather-resistant and breathable soft-shell pants keep stride on every mountain endeavor. The midweight stretch-woven polyester won't restrict your moves, and the brushed interior maintains next-to-skin comfort. A 2-way zippered fly keeps things easy with a harness on, and the gusseted, zippered cuffs have 2-position snaps and tie-down loops. With water-resistant zippered pockets: two front, two thigh, one back hip. Recyclable through the Common Threads Recycling Program.","[Durable stretch-woven polyester (47% recycled) with DWR (durable water repellent) finish is water- and wind-resistant and highly breathable, Brushed interior for next-to-skin comfort, Waistband has belt loops and elastic on back, 2-way zippered fly, ""External pockets: two front slash, two glued-on thigh, one back hip - all with highly water-resistant, DWR-finished zippers"", ""Gusseted, zippered cuff with 2-position adjustable settings and tie-down loops""]","""7.6-oz 90-denier 92% polyester (47% recycled)/8% spandex, with Deluge DWR finish. Recyclable through the Common Threads Recycling Program""",(597 g 20.7 oz),Vietnam
4,"Alpine w

In [13]:
sample_clean["details"] = sample_clean["details"].apply(
    lambda list: " ".join(list)
)

In [14]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s%/.,-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

In [15]:
for col in ["description_text", "details", "fabric", "weight", "country"]:
    sample_clean[col] = sample_clean[col].apply(
        lambda text: clean_text(text) if pd.notna(text) else text
    )

In [16]:
sample_clean.shape

(500, 7)

In [17]:
display(sample_clean)

,id,description,description_text,details,fabric,weight,country
0,1,"Active classic boxers - There's a reason why our boxers are a cult favorite - they keep their cool, especially in sticky situations. The quick-drying, lightweight underwear takes up minimal space in a travel pack. An exposed, brushed waistband offers next-to-skin softness, five-panel construction with a traditional boxer back for a classic fit, and a functional fly. Made of 3.7-oz 100% recycled polyester with moisture-wicking performance. Inseam (size M) is 4 1/2"". Recyclable through the Common Threads Recycling Program.<br><br><b>Details:</b><ul> <li>""Silky Capilene 1 fabric is ultralight, breathable and quick-to-dry""</li> <li>""Exposed, brushed elastic waistband for comfort""</li> <li>5-panel construction with traditional boxer back</li> <li>""Inseam (size M) is 4 1/2""""""</li></ul><br><br><b>Fabric: </b>3.7-oz 100% all-recycled polyester with Gladiodor natural odor control for the garment. Recyclable through the Common Threads Recycling Program<br><br><b>Weight: </b>99 g (3.5 oz)<br><br>Made in Mexico.","active classic boxers - there s a reason why our boxers are a cult favorite - they keep their cool, especially in sticky situations. the quick-drying, lightweight underwear takes up minimal space in a travel pack. an exposed, brushed waistband offers next-to-skin softness, five-panel construction with a traditional boxer back for a classic fit, and a functional fly. made of 3.7-oz 100% recycled polyester with moisture-wicking performance. inseam size m is 4 1/2 . recyclable through the common threads recycling program.","silky capilene 1 fabric is ultralight, breathable and quick-to-dry exposed, brushed elastic waistband for comfort 5-panel construction with traditional boxer back inseam size m is 4 1/2",3.7-oz 100% all-recycled polyester with gladiodor natural odor control for the garment. recyclable through the common threads recycling program,99 g 3.5 oz,mexico
1,2,"Active sport boxer briefs - Skinning up Glory requires enough movement without your boxers deciding to poach their own route. The form-fitting Active Sport Boxer Briefs are made from breathable 93% polyester (71% recycled) fabric that's fast-wicking, dries quickly and has 7% spandex for stretch; the seamless waistband and soft leg edges won't roll or bind. The gusseted, flat-sewn 6"" inseam (size M) is offset to prevent inner-thigh chafe. Fly-free with a smooth front panel. Recyclable through the Common Threads Recycling Program.<br><br><b>Details:</b><ul> <li>""Stretch mesh provides support, open-weave mesh for airflow, wicks efficiently, dries fast""</li> <li>Seamless construction</li> <li>""Flat-sewn, gusseted inseam is set forward to prevent inner-thigh chafe""</li> <li>Fly-free support</li> <li>""Inseam (size M) is 6""""""</li></ul><br><br><b>Fabric: </b>""4.6-oz 93% polyester (71% recycled)/7% spandex, with moisture-wicking performance. Recyclable through the Common Threads Recycling Program""<br><br><b>Weight: </b>(60 g 2.1 oz)<br><br>Made in Israel.","active sport boxer briefs - skinning up glory requires enough movement without your boxers deciding to poach their own route. the form-fitting active sport boxer briefs are made from breathable 93% polyester 71% recycled fabric that s fast-wicking, dries quickly and has 7% spandex for stretch the seamless waistband and soft leg edges won t roll or bind. the gusseted, flat-sewn 6 inseam size m is offset to prevent inner-thigh chafe. fly-free with a smooth front panel. recyclable through the common threads recycling program.","stretch mesh provides support, open-weave mesh for airflow, wicks efficiently, dries fast seamless construction flat-sewn, gusseted inseam is set forward to prevent inner-thigh chafe fly-free support inseam size m is 6","4.6-oz 93% polyester 71% recycled /7% spandex, with moisture-wicking performance. recyclable through the common threads recycling program",60 g 2.1 oz,israel
2,3,"Active sport briefs - These supe

In [18]:
sample_clean.drop(columns=["description"], inplace=True)

In [19]:
sample_clean

,id,description_text,details,fabric,weight,country
0,1,"active classic boxers - there s a reason why our boxers are a cult favorite - they keep their cool, especially in sticky situations. the quick-drying, lightweight underwear takes up minimal space in a travel pack. an exposed, brushed waistband offers next-to-skin softness, five-panel construction with a traditional boxer back for a classic fit, and a functional fly. made of 3.7-oz 100% recycled polyester with moisture-wicking performance. inseam size m is 4 1/2 . recyclable through the common threads recycling program.","silky capilene 1 fabric is ultralight, breathable and quick-to-dry exposed, brushed elastic waistband for comfort 5-panel construction with traditional boxer back inseam size m is 4 1/2",3.7-oz 100% all-recycled polyester with gladiodor natural odor control for the garment. recyclable through the common threads recycling program,99 g 3.5 oz,mexico
1,2,"active sport boxer briefs - skinning up glory requires enough movement without your boxers deciding to poach their own route. the form-fitting active sport boxer briefs are made from breathable 93% polyester 71% recycled fabric that s fast-wicking, dries quickly and has 7% spandex for stretch the seamless waistband and soft leg edges won t roll or bind. the gusseted, flat-sewn 6 inseam size m is offset to prevent inner-thigh chafe. fly-free with a smooth front panel. recyclable through the common threads recycling program.","stretch mesh provides support, open-weave mesh for airflow, wicks efficiently, dries fast seamless construction flat-sewn, gusseted inseam is set forward to prevent inner-thigh chafe fly-free support inseam size m is 6","4.6-oz 93% polyester 71% recycled /7% spandex, with moisture-wicking performance. recyclable through the common threads recycling program",60 g 2.1 oz,israel
2,3,"active sport briefs - these superbreathable no-fly briefs are the minimalist s choice for high-octane endeavors. made from a blend of fast-wicking, quick-drying 93% polyester 71% recycled and 7% spandex that has both stretch-mesh for support and open mesh for cooling airflow . soft edging at the leg openings and a seamless waist won t roll or create friction against layers. with a smooth front panel for opacity. recyclable through the common threads recycling program.","stretch mesh provides support, open-weave mesh for airflow, wicks efficiently and dries fast seamless construction leg openings have half-inch self-binding for comfort fly-free support","4.6-oz 93% polyester 71% recycled /7% spandex, with moisture-wicking performance. recyclable through the common threads recycling program",49 g 1.7 oz,israel
3,4,"alpine guide pants - skin in, climb ice, switch to rock, traverse a knife-edge ridge and boogie back down - these durable, weather-resistant and breathable soft-shell pants keep stride on every mountain endeavor. the midweight stretch-woven polyester won t restrict your moves, and the brushed interior maintains next-to-skin comfort. a 2-way zippered fly keeps things easy with a harness on, and the gusseted, zippered cuffs have 2-position snaps and tie-down loops. with water-resistant zippered pockets two front, two thigh, one back hip. recyclable through the common threads recycling program.","durable stretch-woven polyester 47% recycled with dwr durable water repellent finish is water- and wind-resistant and highly breathable brushed interior for next-to-skin comfort waistband has belt loops and elastic on back 2-way zippered fly external pockets two front slash, two glued-on thigh, one back hip - all with highly water-resistant, dwr-finished zippers gusseted, zippered cuff with 2-position adjustable settings and tie-down loops","7.6-oz 90-denier 92% polyester 47% recycled /8% spandex, with deluge dwr finish. recyclable through the common threads recycling program",597 g 20.7 oz,vietnam
4,5,"alpine wind jkt - on high ridges, steep ice and anything alpine, this jacket serves as a true best of all worl

In [22]:
import spacy
from spacy.lang.char_classes import HYPHENS
from spacy.util import compile_infix_regex
nlp = en_core_web_sm.load()

infixes = [x for x in nlp.Defaults.infixes if HYPHENS not in x]
nlp.tokenizer.infix_finditer = compile_infix_regex(infixes).finditer

for col in ["description_text", "details", "fabric", "weight", "country"]:
    sample_clean[col] = sample_clean[col].apply(
        lambda text: " ".join(
            token.lemma_ for token in nlp(text) if not token.is_stop
        ) if pd.notna(text) else text
    )
print([t.text for t in nlp("united-states")])  # doit afficher ['t-shirt']

['united-states']


In [23]:
for col in ["description_text", "details", "fabric", "weight", "country"]:
    print(f"\n--- {col} ---")
    for text in sample_clean[col].dropna():
        doc = nlp(text)
        print("Tokens:", [token.text for token in doc])
        print("Lemma:", [token.lemma_ for token in doc])
        print("Indices:", [token.i for token in doc])
        print("Alpha Numeric:", [token.is_alpha for token in doc])
        print("Numbers:", [token.like_num for token in doc])
        print("Stopwords restants:", [token.text for token in doc if token.is_stop])
        print("------------")


--- description_text ---
Tokens: ['active', 'classic', 'boxer', '-', 's', 'reason', 'boxer', 'cult', 'favorite', '-', 'cool', ',', 'especially', 'sticky', 'situation', '.', 'quick-drying', ',', 'lightweight', 'underwear', 'minimal', 'space', 'travel', 'pack', '.', 'expose', ',', 'brush', 'waistband', 'offer', 'next-to-skin', 'softness', ',', 'five-panel', 'construction', 'traditional', 'boxer', 'classic', 'fit', ',', 'functional', 'fly', '.', '3.7-oz', '100', '%', 'recycle', 'polyester', 'moisture-wicking', 'performance', '.', 'inseam', 'size', 'm', '4', '1/2', '.', 'recyclable', 'common', 'thread', 'recycle', 'program', '.']
Lemma: ['active', 'classic', 'boxer', '-', 's', 'reason', 'boxer', 'cult', 'favorite', '-', 'cool', ',', 'especially', 'sticky', 'situation', '.', 'quick-drying', ',', 'lightweight', 'underwear', 'minimal', 'space', 'travel', 'pack', '.', 'expose', ',', 'brush', 'waistband', 'offer', 'next-to-skin', 'softness', ',', 'five-panel', 'construction', 'traditional', 'b

In [24]:
print(STOP_WORDS)

{'my', 'still', 'herein', 'such', 'will', 'them', 'mine', 'of', 'own', 'used', 'seem', 'until', 'whoever', 'had', 'their', 'we', 'while', 'to', 'your', '‘d', 'amongst', 'wherein', "'s", 'becoming', 'everything', 'hers', 'formerly', 'anyhow', 'why', 'but', 'thence', 'yours', 'various', 'nowhere', 'therein', 'everyone', 'there', 'else', '‘m', 'being', "n't", 'unless', 'also', 'side', 'serious', 'say', 'always', 'noone', 'if', 'less', 'hereafter', 'only', 'beyond', 'into', 'from', 'both', 'anywhere', 'nobody', 'others', 'after', 'few', 'us', 'have', 'that', 'back', 'so', 'everywhere', 'third', 'its', 'in', '’m', 'per', 'more', 'most', 'onto', 'just', '’ve', 'for', 'either', 'amount', 'five', 'fifteen', 'must', 'was', 'yet', 'before', 'whereafter', 'below', 'none', '’ll', "'d", 'our', 'each', 'since', 'again', 'meanwhile', 'even', 'me', 'thereby', 'himself', 'be', '‘s', 'one', 'does', 'am', 'his', 'became', 'are', 'once', '‘re', 'six', 'yourself', 'ten', 'ours', 'i', 'about', 'may', 'sever

In [25]:
display(sample_clean)

,id,description_text,details,fabric,weight,country
0,1,"active classic boxer - s reason boxer cult favorite - cool , especially sticky situation . quick-drying , lightweight underwear minimal space travel pack . expose , brush waistband offer next-to-skin softness , five-panel construction traditional boxer classic fit , functional fly . 3.7-oz 100 % recycle polyester moisture-wicking performance . inseam size m 4 1/2 . recyclable common thread recycle program .","silky capilene 1 fabric ultralight , breathable quick-to-dry expose , brush elastic waistband comfort 5-panel construction traditional boxer inseam size m 4 1/2",3.7-oz 100 % all-recycled polyester gladiodor natural odor control garment . recyclable common thread recycle program,99 g 3.5 oz,mexico
1,2,"active sport boxer brief - skin glory require movement boxer decide poach route . form-fitting active sport boxer brief breathable 93 % polyester 71 % recycle fabric s fast-wicking , dry quickly 7 % spandex stretch seamless waistband soft leg edge win t roll bind . gussete , flat-sewn 6 inseam size m offset prevent inner-thigh chafe . fly-free smooth panel . recyclable common thread recycle program .","stretch mesh provide support , open-weave mesh airflow , wick efficiently , dry fast seamless construction flat-sewn , gussete inseam set forward prevent inner-thigh chafe fly-free support inseam size m 6","4.6-oz 93 % polyester 71 % recycle /7 % spandex , moisture-wicking performance . recyclable common thread recycle program",60 g 2.1 oz,israel
2,3,"active sport brief - superbreathable no-fly brief minimalist s choice high-octane endeavor . blend fast-wicking , quick-drye 93 % polyester 71 % recycle 7 % spandex stretch-mesh support open mesh cool airflow . soft edge leg opening seamless waist win t roll create friction layer . smooth panel opacity . recyclable common thread recycle program .","stretch mesh provide support , open-weave mesh airflow , wick efficiently dry fast seamless construction leg opening half-inch self-binde comfort fly-free support","4.6-oz 93 % polyester 71 % recycle /7 % spandex , moisture-wicking performance . recyclable common thread recycle program",49 g 1.7 oz,israel
3,4,"alpine guide pant - skin , climb ice , switch rock , traverse knife-edge ridge boogie - durable , weather-resistant breathable soft-shell pant stride mountain endeavor . midweight stretch-woven polyester win t restrict , brush interior maintain next-to-skin comfort . 2-way zipper fly thing easy harness , gussete , zippered cuff 2-position snap tie-down loop . water-resistant zippered pocket , thigh , hip . recyclable common thread recycle program .","durable stretch-woven polyester 47 % recycle dwr durable water repellent finish water- wind-resistant highly breathable brush interior next-to-skin comfort waistband belt loop elastic 2-way zipper fly external pocket slash , glued-on thigh , hip - highly water-resistant , dwr-finished zipper gussete , zippered cuff 2-position adjustable set tie-down loop","7.6-oz 90-denier 92 % polyester 47 % recycle /8 % spandex , deluge dwr finish . recyclable common thread recycle program",597 g 20.7 oz,vietnam
4,5,"alpine wind jkt - high ridge , steep ice alpine , jacket serve true good world staple . excel stand-alone shell blustery rock climb , cool-weather trail run high-output ski tour . , condition ice alpine climbing , function lightly insulate windshirt approach , frictionless midlayer s time bundle tie-in . polyester ripstop shell deluge dwr durable water repellent treatment shed snow block wind , smooth , lightly brush hang mesh liner wick moisture , dry fast , doesn t bind baselayer . superlight stretch-woven underarm panel enhance breathability allow unimpaired arm motion , hand pocket close zipper . drawcord hem , elastic cuff , heat-transfer reflective logo regular-coil , center-front zipper dwr finish round feature . update season improve fit . recyclable common thread recycle program .","lightweight , breathable polyester rip

In [26]:
sample_clean.shape

(500, 6)

In [27]:
sample_clean.to_csv(r'C:\Users\LN\Documents\Jedha\Projets\NORTH_FACE\Helene\data\sample_clean.csv', index=False)